# ANTECIPA — Pipeline de dados reais do PNCP no Google Colab

Projeto de dissertação (FACE/UnB): gestão preditiva de riscos em compras públicas no STF.

Este notebook roda o pipeline direto da pasta do projeto no seu Google Drive
(`unb/antecipa-real/`), reutilizando os caches de coleta já existentes em
`dados/brutos/` — ou seja, **não é preciso recoletar nada** para treinar os
modelos; a coleta só roda de novo se você apagar os JSONs de cache.

Ordem das seções: (1) montar o Drive · (2) localizar o projeto ·
(3) dependências · (4) coleta (opcional, usa cache) · (5) datasets e treino
v1/v2 · (6) publicação do modelo calibrado + dashboard.

In [1]:
# 1. Montar o Google Drive (vai pedir autorização da conta ddnprata@gmail.com)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Localizar a pasta do projeto no Drive montado.
# A pasta unb está sincronizada via "Outros computadores/Meu laptop", que o
# Colab expõe em /content/drive/Othercomputers/. O glob abaixo encontra o
# pipeline onde quer que esteja (MyDrive ou Othercomputers).
from pathlib import Path

candidatos = list(Path('/content/drive').glob('**/antecipa-real/pipeline/coleta_pncp.py'))
assert candidatos, 'Pasta antecipa-real não encontrada no Drive — confira a sincronização.'
PIPELINE = candidatos[0].parent
print('Projeto encontrado em:', PIPELINE)
%cd {str(PIPELINE)}

Projeto encontrado em: /content/drive/Othercomputers/Meu laptop/Meu Drive (ddnprata@gmail.com)/unb/antecipa-real/pipeline
/content/drive/Othercomputers/Meu laptop/Meu Drive (ddnprata@gmail.com)/unb/antecipa-real/pipeline


In [ ]:
# 3. Dependências (o Colab já traz scikit-learn; o comando só garante a versão)
%pip install -q scikit-learn

## 4. Coleta (opcional)

Os scripts são **idempotentes**: se o JSON de cache existe em `dados/brutos/`,
a etapa é pulada em segundos. Para recoletar algo do zero, apague o arquivo
correspondente antes de rodar.

⚠️ A coleta completa do zero leva ~40–60 min e sofre rate-limit do PNCP
(HTTP 429) — por isso existe o passe de reparo. Com os caches do Drive, a
célula abaixo termina quase instantaneamente.

In [ ]:
!python coleta_pncp.py        # STF: contratos, termos, fornecedores, itens
!python coleta_precos_v2.py   # preços de referência (PDM/CATMAT)
!python coleta_multi.py       # 10 tribunais federais: contratos + termos
!python repara_multi.py       # refaz janelas que falharam por rate-limit
!python coleta_cadastros.py   # Receita Federal p/ fornecedores únicos

## 5. Datasets e treino (v1 e v2)

- **v1**: rótulo de desfecho adverso sem horizonte (relatorio_modelo.md)
- **v2**: rótulo com horizonte fixo de 12 meses — corrige a censura
  (relatorio_modelo_v2.md)
- Experimento extra: treino só-STF × multi-órgão no mesmo teste

In [ ]:
!python monta_dataset.py
!python treina_modelo.py
!python monta_dataset_v2.py
!python treina_modelo_v2.py
!python treina_stf_vs_multi.py

In [ ]:
# Ver os relatórios gerados
from IPython.display import Markdown, display
base = PIPELINE.parent / 'dados' / 'processados'
display(Markdown((base / 'relatorio_modelo_v2.md').read_text(encoding='utf-8')))

## 6. Publicar modelo calibrado e regenerar o dashboard

O HTML final é autocontido e fica salvo **de volta no Drive**, na pasta `unb/`
(`ANTECIPA_dashboard_real.html`) — sincroniza automaticamente com o laptop.

In [ ]:
!python publica_modelo_v2.py
!python processa.py
!python gera_dashboard.py

In [ ]:
# Visualizar o dashboard dentro do próprio Colab (ou baixar o arquivo)
from IPython.display import HTML, display
html = (PIPELINE.parent.parent / 'ANTECIPA_dashboard_real.html').read_text(encoding='utf-8')
display(HTML(f'<iframe srcdoc="{html.replace(chr(34), "&quot;")}" width="100%" height="800"></iframe>'))

---
### Notas

- **Escrita no Drive**: tudo que os scripts gravam (caches, datasets, relatórios,
  dashboard) vai para a própria pasta do projeto no Drive — as alterações
  aparecem no laptop pela sincronização.
- **Se o glob da célula 2 não encontrar o projeto**: no Colab, abra o painel de
  arquivos (ícone de pasta à esquerda) e navegue em `drive/` para descobrir o
  caminho real; ajuste `PIPELINE` manualmente.
- **Sessões do Colab expiram** (~12 h): como o estado fica no Drive, basta
  reabrir o notebook e rodar de novo — os caches retomam de onde parou.
- Documentação completa do pipeline: `antecipa-real/README.md`.